# Experiment 6: User-based and item-based collaborative filtering

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Rows = users, columns = items, 0 means "not rated"
ratings = np.array([
    [5, 3, 4, 4, 0],
    [3, 1, 2, 3, 3],
    [4, 3, 4, 3, 5],
    [3, 3, 1, 5, 4],
    [1, 5, 5, 2, 1],
    [0, 0, 0, 4, 4],
], dtype=float)
users = [f"U{i+1}" for i in range(ratings.shape[0])]
items = [f"I{j+1}" for j in range(ratings.shape[1])]
rated = ratings > 0

print(pd.DataFrame(ratings, index=users, columns=items))

user_sim = cosine_similarity(ratings)     # user x user
item_sim = cosine_similarity(ratings.T)   # item x item

In [ ]:
def predict_user_based(u, i):
    # Weighted average of ratings for item i from users similar to u
    others = np.where(rated[:, i])[0]
    others = others[others != u]
    w = user_sim[u, others]
    denom = np.abs(w).sum()
    return np.dot(w, ratings[others, i]) / denom if denom else 0.0

def predict_item_based(u, i):
    # Weighted average of user u's ratings on items similar to i
    mine = np.where(rated[u])[0]
    mine = mine[mine != i]
    w = item_sim[i, mine]
    denom = np.abs(w).sum()
    return np.dot(w, ratings[u, mine]) / denom if denom else 0.0

def recommend(u, method, n=3):
    predict = predict_user_based if method == "user" else predict_item_based
    scores = {i: predict(u, i) for i in range(len(items)) if not rated[u, i]}
    top = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:n]
    return [(items[i], round(s, 2)) for i, s in top]

for u in range(len(users)):
    print(users[u], "| user-based:", recommend(u, "user"), "| item-based:", recommend(u, "item"))